In [1]:
import torch
import re
import jieba
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import time

D:\Anaconda\envs\stock1\lib\site-packages\torch\cuda\__init__.py:83: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
D:\Anaconda\envs\stock1\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
#获取数据，进行分词，获取词表
#数据预处理，构建数据集
#搭建RNN
#训练模型
#模型预测

In [3]:
def build_vocab():
    #定义变量，记录去重后所有的词，每行文本分词结果
    unique_words,all_words = [],[]
    #遍历数据集，获取每行文本
    for line in open('jaychou_lyrics.txt','r',encoding = 'utf-8'):
        #获取每行数据。进行分词
        words = jieba.lcut(line)
        #print(f'每行数据:{words}')  #每行数据:['想要', '有', '直升机', '\n']
        #所有分词结果记录到all_words中：
        all_words.append(words)    #列表套列表  [[...],[...],...]
        #遍历分词结果，去重后，添加到unique_words中
        for word in words:
            if word not in unique_words:
                unique_words.append(word)
    #统计语料中去重后词的数量
    word_count = len(unique_words)
    #构建词表，字典形式，key是词，value是词的索引
    word_to_index = {word:i for i,word in enumerate(unique_words)}
    #歌词文本一年词表索引表示
    corpus_idx = []
    #遍历每一行的分词结果
    for words in all_words:
        #定义遍历，记录词索引列表。
        tmp = []
        #获取每一行的词，并获取相应的索引
        for word in words:
            tmp.append(word_to_index[word])
        #在每行词之间，添加空格隔开。
        tmp.append(word_to_index[' '])
        #获取文档中每个词的索引，添加到corpus_idx中
        corpus_idx.extend(tmp)
    #返回结果，唯一词表（5702个词），词表，（去重后词的数量），歌词文本用词表索引表示
    return unique_words,word_to_index,word_count,corpus_idx

    


In [4]:
unique_words,word_to_index,word_count,corpus_idx = build_vocab()
print(f'词的数量:{word_count}')
print(f'去重后的词:{unique_words}')
print(f'每个词的索引:{word_to_index}')
print(f'歌词文本索引表示:{corpus_idx}')

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\LIHUIT~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.615 seconds.
Prefix dict has been built successfully.


词的数量:5703
去重后的词:['想要', '有', '直升机', '\n', '和', '你', '飞到', '宇宙', '去', '融化', '在', '一起', '里', '我', '每天', '想想', '著', '这样', '的', '甜蜜', '让', '开始', '相信', '命运', '感谢', '地心引力', '碰到', '漂亮', '面红', '可爱', '女人', '温柔', '心疼', '透明', '感动', '坏坏', '疯狂', '乡', '如果说', '怀疑', ' ', '可以', '造句', '分离', '能够', '翻译', '如果', '这', '一切', '真的', '将', '寂寞', '封闭', '然后', '这里', '不', '限', '日期', '过去', '慢慢', '温习', '爱上你', '那场', '悲剧', '是', '完美', '演出', '一场', '戏', '宁愿', '心碎', '哭泣', '再', '狠狠', '忘记', '爱过', '证据', '晶莹', '泪滴', '闪烁', '成', '回忆', '伤人', '美丽', '完美主义', '太', '彻底', '连', '恨', '都', '难以', '下笔', '真心', '抽离', '写成', '日记', '像是', '默剧', '分手', '的话', '像', '语言', '暴力', '已', '无能为力', '提起', '决定', '中断', '熟悉', '周杰伦', '一步', '两步', '三步', '四步', '望', '著天', '看', '星星', '一颗', '两颗', '三颗', '四颗', '连成线', '乘著风', '游荡', '蓝天', '边', '一片', '云', '掉落在', '面前', '捏成', '形状', '随风', '跟', '一口', '吃掉', '忧愁', '载著', '彷', '彿', '载', '阳光', '不管', '到', '哪里', '晴天', '蝴蝶', '自在', '飞', '花', '也', '布满', '天', '一朵', '因', '而', '香', '试图', '夕阳', '飞翔', '带领', '环绕', '大自然', '迎著风', '共渡', '每', '一天', '手牵

In [11]:
#构建数据data->张量tensor->数据集对象dataset->数据加载器dataloader

In [6]:
class LyricsDataset(torch.utils.data.Dataset):
    def __init__(self,corpus_idx,num_chars):
        #文档数据中的索引
        self.corpus_idx = corpus_idx
        #每个句子中词的个数
        self.num_chars = num_chars
        #文档数据中词的数量
        self.word_count = len(self.corpus_idx)
        #句子数量
        self.number = self.word_count // self.num_chars
    # 当使用len(obj)时，自动调用本方法
    def __len__(self):
        return self.number
    def __getitem__(self,idx):
        #idx:指的是词的索引，并将其修正索引值到文档的范围内。
        #确保索引start在合法范围内，避免越界。start:当前样本的起始索引 idx*num_chars：当前句子的起始索引。
        start = min(max(idx,0),self.word_count - self.num_chars - 1)
        end = start + self.num_chars
        #输入值：从文档中取出start - end的索引的词，作为x
        x = self.corpus_idx[start:end]
        #输出值
        y = self.corpus_idx[start+1:end+1]
        #返回输入值和输出值
        return torch.tensor(x),torch.tensor(y)

In [7]:
dataset = LyricsDataset(corpus_idx,5)
print(len(dataset))

9827


In [7]:
x,y = dataset[1]
print(x)
print(y)

tensor([ 1,  2,  3, 40,  0])
tensor([ 2,  3, 40,  0,  4])


In [10]:
import torch.nn as nn

In [25]:
#搭建RNN神经网络
class TextGenerator(nn.Module):
    def __init__(self,unique_word_count):
        #初始化父类成员
        super().__init__()
        #初始化词嵌入层，语料中词的数量，词向量的维度
        self.ebd = nn.Embedding(unique_word_count,embedding_dim=120)
        #循环网络层：词向量维度：120，隐藏层维度：256，网络层数：1
        self.rnn = nn.RNN(120,256,1)
        # 输出层：特征向量维度，词表中词的个数
        self.out = nn.Linear(256,unique_word_count)   #词表中每个词的频率，选概率最大的那个词作为预测结果。
    def forward(self,inputs,hidden):
        #初始化词嵌入层
        embd = self.ebd(inputs)
        #RNN处理
        output,hidden = self.rnn(embd.transpose(0,1),hidden)
        #全连接，输入内容必须是二维数据，即：词的数量*词的维度
        #输入维度：（seq_len句子数量 * batch，词向量维度256）
        #输出维度：（seq_len句子数量 *batch,词表中词的个数）
        output = self.out(output.reshape(-1, output.shape[-1]))
        return output,hidden
    def init_hidden(self,bs):
        #隐藏层初始化：[网络层数，batch，隐藏层的向量维度]
        return torch.zeros(1,bs,256)


In [29]:
#训练模型
def train():
    unique_words,word_to_index,unique_word_count,corpus_idx = build_vocab()
    lyrics = LyricsDataset(corpus_idx,5)
    model = TextGenerator(unique_word_count)
    lyrics_dataloader = DataLoader(lyrics,batch_size =5,shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr = 0.01)
    epochs = 10
    for epoch in range(epochs): #epoch:0，1，2，3，。。9第1轮第2轮
        start,iter_num,total_loss = time.time(),0,0.0
        for x,y in lyrics_dataloader:
            current_batch_size = x.size(0)
            hidden = model.init_hidden(current_batch_size)
            #模型计算：
            output,hidden = model(x,hidden)
            #计算损失
            #y的形状 (batch,seq_len,词向量维度) →转向一维向量 → 每个词的下标索引
            y = torch.transpose(y,0,1).reshape(-1,)
            loss = criterion(output,y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            iter_num += 1
    torch.save(model.state_dict(),f'text_generator.pth')  
            
        

In [30]:
train()

In [33]:
#模型预测
def evaluate(start_word,sentence_length):
    #1、构建词典
    unique_words,word_to_index,unique_word_count,corpus_idx = build_vocab()
    #2. 获取模型
    model = TextGenerator(unique_word_count)
    #3、加载模型参数
    model.load_state_dict(torch.load('text_generator.pth'))
    #4、获取隐藏层初始值
    hidden = model.init_hidden(1)
    #5、将输入的开始词转化为索引
    word_idx = word_to_index[start_word]
    #6、定义列表，存放：产生的词的索引
    generate_sentence = [word_idx] #开始词的索引
    #7、遍历句子长度，获取到每一个词
    for i in range(sentence_length):
        #7、1 模型预测
        output,hidden = model(torch.tensor([[word_idx]]),hidden)
        #获取预测结果,argmax（）从所有结果中找最大值对应的索引
        word_idx = torch.argmax(output)
        #把预测结果添加到列表中
        generate_sentence.append(word_idx)

    for idx in generate_sentence:
        print(unique_words[idx],end='')
        
        
        
    

In [36]:
evaluate('再见',50)

再见感动的可爱女人
 坏坏的笑容 有何不同 还是朋友
 娘子依旧每日折一枝杨柳
 你在操纵
 大峡谷的风呼啸而过
 坏坏的让我心疼的可爱女人
 难道你手不会